In [1]:
"""
Weight Sharing — Circulant / Structured Matrices
=================================================

MATHEMATICAL FOUNDATION
------------------------

1. CIRCULANT MATRIX
   A circulant matrix circ(c) for c = [c₀, c₁, ..., c_{b-1}] ∈ ℝᵇ is:

       ┌ c₀   c_{b-1} … c₁  ┐
       │ c₁   c₀      … c₂  │
       │  ⋮    ⋮       ⋱  ⋮  │
       └ c_{b-1} c_{b-2} … c₀┘

   Element (i, j) = c_{(j-i) mod b}.
   Only b values needed to represent a b×b matrix → b× compression per block.

2. BLOCK CIRCULANT APPROXIMATION (VECTORIZED)
   Given weight matrix W ∈ ℝ^{m×n}, pad to (mp × np) where mp, np are the
   nearest multiples of b, then reshape into (N, b, b) blocks via:

       W_pad.reshape(bm, b, bn, b).permute(0, 2, 1, 3).reshape(N, b, b)

   For each block B, the optimal c under ‖B − circ(c)‖²_F is the diagonal
   average:

       c_k = (1/b) Σᵢ B[i, (i+k) mod b]

   computed in one batched gather+mean — no Python loop over blocks.

3. PARAMETER SAVINGS
   - Original block: b² parameters
   - Compressed block: b parameters
   - Ratio: 1/b  (e.g. b=8 → 87.5% reduction per full block)
   - Padding zeros are discarded after reconstruction (lossless for real data).

4. CONV LAYER RESHAPE
   Conv weight W ∈ ℝ^{C_out × C_in × kH × kW} is treated as a 2D matrix:

       W_2d ∈ ℝ^{C_out × (C_in · kH · kW)}

   then block-circulant compressed.

5. CIRCULANT FAST MULTIPLY (FFT, informational)
   circ(c) · x = IFFT(FFT(c) ⊙ FFT(x)), reducing O(b²) to O(b log b).
   Not implemented here — parameter savings are real regardless.

6. WEIGHT CACHE STRATEGY
   - model.eval()  → cache reconstructed weight on first forward call; reuse.
   - model.train() → NEVER cache; rebuild every forward call for correct autograd.

OUTPUT
======
  __3__Circulating_Matrices.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch : {torch.__version__}", flush=True)
print(f"[init] Device  : {DEVICE}", flush=True)

# ── Config ─────────────────────────────────────────────────────────────────────
ORIGINAL_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
MODEL_ARCH            = "resnet50"
OUTPUT_JSON           = "__3__Circulating_Matrices.json"

NUM_CLASSES  = 10
BATCH_SIZE   = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS  = 0
PIN_MEMORY   = DEVICE.type == "cuda"

BLOCK_SIZE      = 8      # b — compression ratio per block = 1/b
FINETUNE_EPOCHS = 5
FINETUNE_LR     = 1e-3

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] BlockSize: {BLOCK_SIZE}  FineTuneEpochs: {FINETUNE_EPOCHS}", flush=True)


# ══════════════════════════════════════════════════════════════════════════════
# ── Model / Data helpers ──────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def build_model(arch="resnet50", num_classes=10):
    m = models.resnet50(weights=None) if arch == "resnet50" else models.resnet18(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m


def load_model(path, arch):
    print(f"[model] Loading {arch} from {path} ...", flush=True)
    m = build_model(arch, NUM_CLASSES)
    m.load_state_dict(torch.load(path, map_location="cpu"))
    return m.to(DEVICE).eval()


def get_loaders():
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    train_ds = torchvision.datasets.CIFAR10("../data", train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10("../data", train=False, download=True, transform=test_tf)
    return (
        DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
        DataLoader(test_ds,  512,        shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
    )


def model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)


def param_count(model):
    return sum(p.numel() for p in model.parameters())


# ══════════════════════════════════════════════════════════════════════════════
# ── Evaluation / Profiling ────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def evaluate(model, loader):
    """Evaluate in eval mode. Caches are warm here — safe and fast."""
    model.eval()
    _invalidate_all_caches(model)
    preds, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        preds.extend(model(x).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
    return {
        "accuracy" : float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall"   : float(recall_score(labels,    preds, average="macro", zero_division=0)),
        "f1"       : float(f1_score(labels,         preds, average="macro", zero_division=0)),
    }


def measure_inference_ms(model, device, batch_size=128, runs=20):
    model.eval().to(device)
    _invalidate_all_caches(model)
    dummy = torch.randn(batch_size, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(3):
            model(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs):
            model(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs * 1000


def count_flops(model, device, input_size=(1, 3, 32, 32)):
    """FLOPs via FlopCounterMode (PyTorch ≥ 2.1), falling back to hook-based counting."""
    model = model.eval().to(device)
    _invalidate_all_caches(model)
    dummy = torch.randn(*input_size, device=device)
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with FlopCounterMode(model, display=False) as fcm:
            with torch.no_grad():
                model(dummy)
        return fcm.get_total_flops()
    except Exception:
        pass

    # Fallback: manual hooks
    total = [0]

    def conv_hook(module, inp, out):
        C_out, C_in, kH, kW = module.weight.shape
        _, _, H_out, W_out  = out.shape
        total[0] += 2 * C_in * kH * kW * C_out * H_out * W_out // module.groups

    def linear_hook(module, inp, out):
        total[0] += 2 * module.in_features * module.out_features

    hooks = []
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, CirculantConv)):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, (nn.Linear, CirculantLinear)):
            hooks.append(m.register_forward_hook(linear_hook))
    with torch.no_grad():
        model(dummy)
    for h in hooks:
        h.remove()
    return total[0]


# ══════════════════════════════════════════════════════════════════════════════
# ── Circulant Maths — fully vectorized, no Python loops in hot paths ──────────
# ══════════════════════════════════════════════════════════════════════════════

def _make_circ_index(b: int, device) -> torch.Tensor:
    """
    Precompute gather index I ∈ ℤ^{b×b} where I[i,j] = (j−i) mod b.
    Shape: (b, b).
    """
    cols = torch.arange(b, device=device)
    rows = torch.arange(b, device=device)
    return (cols.unsqueeze(0) - rows.unsqueeze(1)) % b   # (b, b)


def _build_circulant_batched(C: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    """
    Expand N vectors of length b into N circulant matrices of shape (b, b).

    Math: out[n, i, j] = C[n, (j−i) mod b] = C[n, idx[i,j]]

    Shapes: C (N, b), idx (b, b) → out (N, b, b).
    """
    N, b    = C.shape
    idx_exp = idx.unsqueeze(0).expand(N, -1, -1)   # (N, b, b)
    C_exp   = C.unsqueeze(1).expand(N, b, b)        # (N, b, b)
    return C_exp.gather(2, idx_exp)                  # (N, b, b)


def _extract_circulant_params(M: torch.Tensor, b: int):
    """
    Vectorized block-circulant projection using reshape + gather — no Python
    loop over blocks.

    Algorithm:
      1. Zero-pad M to (mp × np_) where mp, np_ are the nearest multiples of b.
      2. Reshape into (N, b, b) blocks via reshape/permute.
      3. Compute diagonal average in one batched gather + mean.

    The padding zeros are discarded when the reconstructed matrix is sliced
    back to (m, n) in _reconstruct_matrix — lossless for all real data values.

    Returns:
        circ_vecs  (N, b)       — one compression vector per block
        boundary   []           — always empty (padding approach needs no boundary)
        layout     tuple        — (m, n, mp, np_) for reconstruction
    """
    m, n   = M.shape
    device = M.device
    dtype  = M.dtype

    # Pad to multiples of b
    mp  = (m + b - 1) // b * b
    np_ = (n + b - 1) // b * b

    if mp != m or np_ != n:
        M_pad = torch.zeros(mp, np_, dtype=dtype, device=device)
        M_pad[:m, :n] = M
    else:
        M_pad = M

    bm = mp // b   # number of block rows
    bn = np_ // b  # number of block cols
    N  = bm * bn   # total blocks

    # Extract all (b×b) blocks: reshape to (bm, b, bn, b), permute, flatten N
    blocks = (M_pad
              .reshape(bm, b, bn, b)
              .permute(0, 2, 1, 3)      # (bm, bn, b, b)
              .reshape(N, b, b))        # (N, b, b)

    # Diagonal average: c_k = (1/b) Σᵢ blocks[:, i, (i+k)%b]
    # diag_gi[i, k] = (i + k) % b  — column index for diagonal k at row i
    r_idx   = torch.arange(b, device=device)
    k_idx   = torch.arange(b, device=device)
    diag_gi = (r_idx.unsqueeze(1) + k_idx.unsqueeze(0)) % b   # (b, b)
    diag_gi = diag_gi.unsqueeze(0).expand(N, -1, -1)           # (N, b, b)

    gathered  = blocks.gather(2, diag_gi)   # (N, b, b)
    circ_vecs = gathered.mean(dim=1)        # (N, b) — average over rows

    layout = (m, n, mp, np_)
    return circ_vecs, [], layout


def _reconstruct_matrix(circ_vecs: torch.Tensor,
                         boundary,          # unused — kept for API compatibility
                         layout,
                         out_shape: tuple,
                         b:         int,
                         idx:       torch.Tensor) -> torch.Tensor:
    """
    Reconstruct W ∈ ℝ^{m×n} from compact circulant params.

    All N blocks are expanded in one batched gather call, then folded back
    via reshape/permute. The padded region is sliced off at the end.
    No Python loop over blocks.
    """
    m, n, mp, np_ = layout

    # Expand all N circulant vectors to (N, b, b) matrices
    all_circ = _build_circulant_batched(circ_vecs, idx)   # (N, b, b)

    bm = mp // b
    bn = np_ // b

    # Fold back: (N, b, b) → (bm, bn, b, b) → (bm, b, bn, b) → (mp, np_)
    W_pad = (all_circ
             .reshape(bm, bn, b, b)
             .permute(0, 2, 1, 3)      # (bm, b, bn, b)
             .reshape(mp, np_))

    return W_pad[:m, :n].contiguous()


# ══════════════════════════════════════════════════════════════════════════════
# ── Compressed Layer Classes ──────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

class _CirBase(nn.Module):
    """
    Shared caching logic for circulant layers.

    Cache policy:
      - model.train() → self.training == True  → never cache (autograd safety)
      - model.eval()  → self.training == False → cache on first call, reuse
    """

    def invalidate_cache(self):
        self._weight_cache = None

    def _cached_weight(self, build_fn):
        if self.training:
            return build_fn()
        if self._weight_cache is None:
            with torch.no_grad():
                self._weight_cache = build_fn()
        return self._weight_cache


class CirculantLinear(_CirBase):
    """
    Drop-in replacement for nn.Linear with block-circulant weight compression.

    circ_params stores the flat (N*b,) vector of circulant coefficients.
    No boundary chunks — padding handles edge blocks.
    """

    def __init__(self, src: nn.Linear, block_size: int):
        super().__init__()
        W = src.weight.data       # (out_features, in_features)
        b = block_size

        self.weight_shape = W.shape
        self.block_size   = b
        self.in_features  = src.in_features
        self.out_features = src.out_features

        cv, _, layout         = _extract_circulant_params(W, b)
        self.num_full         = cv.shape[0]
        self.layout           = layout          # (m, n, mp, np_)
        self.register_buffer("_circ_idx", _make_circ_index(b, W.device))

        self.circ_params = nn.Parameter(cv.reshape(-1))   # (N*b,)
        self.bias        = nn.Parameter(src.bias.clone()) if src.bias is not None else None
        self._weight_cache = None

    def _build(self):
        cv = self.circ_params.reshape(self.num_full, self.block_size)
        return _reconstruct_matrix(cv, [], self.layout,
                                   self.weight_shape, self.block_size,
                                   self._circ_idx)

    def forward(self, x):
        return F.linear(x, self._cached_weight(self._build), self.bias)


class CirculantConv(_CirBase):
    """
    Drop-in replacement for nn.Conv2d with block-circulant weight compression.

    The 4-D weight is reshaped to 2-D before compression and back after.
    """

    def __init__(self, src: nn.Conv2d, block_size: int):
        super().__init__()
        W = src.weight.data       # (C_out, C_in, kH, kW)
        b = block_size
        C_out, C_in, kH, kW = W.shape

        self.original_weight_shape = W.shape
        self.matrix_shape          = (C_out, C_in * kH * kW)
        self.block_size            = b
        self.stride                = src.stride
        self.padding               = src.padding
        self.dilation              = src.dilation
        self.groups                = src.groups

        cv, _, layout         = _extract_circulant_params(W.reshape(C_out, -1), b)
        self.num_full         = cv.shape[0]
        self.layout           = layout          # (m, n, mp, np_)
        self.register_buffer("_circ_idx", _make_circ_index(b, W.device))

        self.circ_params = nn.Parameter(cv.reshape(-1))   # (N*b,)
        self.bias        = nn.Parameter(src.bias.clone()) if src.bias is not None else None
        self._weight_cache = None

    def _build(self):
        cv = self.circ_params.reshape(self.num_full, self.block_size)
        W_2d = _reconstruct_matrix(cv, [], self.layout,
                                   self.matrix_shape, self.block_size,
                                   self._circ_idx)
        return W_2d.reshape(self.original_weight_shape)

    def forward(self, x):
        return F.conv2d(x, self._cached_weight(self._build), self.bias,
                        stride=self.stride, padding=self.padding,
                        dilation=self.dilation, groups=self.groups)


# ── Cache management ───────────────────────────────────────────────────────────

def _invalidate_all_caches(model: nn.Module):
    """Flush cached weight tensors in all circulant layers."""
    for m in model.modules():
        if isinstance(m, _CirBase):
            m.invalidate_cache()


# ══════════════════════════════════════════════════════════════════════════════
# ── Model Compression ─────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def _should_compress_conv(layer: nn.Conv2d) -> bool:
    """Skip input conv (C_in=3) and tiny layers smaller than one block."""
    C_out, C_in, kH, kW = layer.weight.shape
    if C_in <= 3:
        return False
    return min(C_out, C_in * kH * kW) >= BLOCK_SIZE


def apply_circulant(model: nn.Module, block_size: int) -> nn.Module:
    model = copy.deepcopy(model)

    def _replace(parent: nn.Module):
        for name, child in list(parent.named_children()):
            if isinstance(child, nn.Conv2d) and _should_compress_conv(child):
                C_out, C_in, kH, kW = child.weight.shape
                print(f"    Circ Conv:   ({C_in}→{C_out}, {kH}×{kW})  D={C_in*kH*kW}", flush=True)
                setattr(parent, name, CirculantConv(child, block_size))
            elif isinstance(child, nn.Linear):
                print(f"    Circ Linear: ({child.in_features}→{child.out_features})", flush=True)
                setattr(parent, name, CirculantLinear(child, block_size))
            else:
                _replace(child)

    _replace(model)
    return model.to(DEVICE)


def count_effective_params(model: nn.Module) -> int:
    """Count only the compressed (circulant) parameters + BN + bias."""
    total = 0
    for m in model.modules():
        if isinstance(m, _CirBase):
            total += m.circ_params.numel()
            if m.bias is not None:
                total += m.bias.numel()
        elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            total += sum(p.numel() for p in m.parameters())
    return total


# ══════════════════════════════════════════════════════════════════════════════
# ── Fine-tuning ───────────────────────────────────────────────────════════════
# ══════════════════════════════════════════════════════════════════════════════

def finetune(model, train_loader, epochs, lr):
    """
    Fine-tune the compressed model.

    model.train() disables the weight cache — every forward call rebuilds
    weight matrices from scratch, preserving a clean autograd graph.
    """
    if epochs == 0:
        return []

    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    accs      = []

    print(f"  Fine-tuning {epochs} epochs @ lr={lr} ...", flush=True)
    for epoch in range(epochs):
        correct = total = 0
        for inputs, targets in train_loader:
            inputs  = inputs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=DEVICE.type,
                                    enabled=(DEVICE.type == "cuda")):
                out  = model(inputs)
                loss = F.cross_entropy(out, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                correct += (out.argmax(1) == targets).sum().item()
                total   += targets.size(0)

        scheduler.step()
        acc = correct / total
        accs.append(round(acc, 6))
        print(f"    FT epoch {epoch+1}/{epochs}  train_acc={acc:.4f}", flush=True)

    return accs


# ══════════════════════════════════════════════════════════════════════════════
# ── Main ──────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def main():
    print(f"\n{'='*65}")
    print("  Weight Sharing — Circulant Structured Matrices")
    print(f"  Model: {MODEL_ARCH}  |  BlockSize: {BLOCK_SIZE}  |  FT: {FINETUNE_EPOCHS} epochs")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline accuracy: {baseline_acc:.4f}\n", flush=True)

    original_model   = load_model(ORIGINAL_MODEL_PATH, MODEL_ARCH)
    original_size_mb = model_size_mb(original_model)
    original_params  = param_count(original_model)
    print(f"  Original: {original_size_mb:.2f} MB  |  Params: {original_params:,}\n", flush=True)

    train_loader, test_loader = get_loaders()

    print("  Counting original FLOPs ...", flush=True)
    original_flops = count_flops(original_model, DEVICE)
    print(f"  Original FLOPs: {original_flops:,}", flush=True)

    # ── Compress ──────────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    print("\n  Applying Circulant compression ...", flush=True)
    circ_model    = apply_circulant(original_model, BLOCK_SIZE)
    compress_time = time.perf_counter() - t0
    print(f"  Done in {compress_time:.1f}s\n", flush=True)

    effective_params = count_effective_params(circ_model)
    param_reduction  = 1.0 - effective_params / original_params
    print(f"  Effective params: {effective_params:,}  "
          f"({effective_params/original_params*100:.1f}% of original)", flush=True)

    pre_ft = evaluate(circ_model, test_loader)
    print(f"  Pre-FT accuracy: {pre_ft['accuracy']:.4f}", flush=True)

    # ── Fine-tune ─────────────────────────────────────────────────────────────
    ft_accs = finetune(circ_model, train_loader, FINETUNE_EPOCHS, FINETUNE_LR)

    # ── Final evaluation ──────────────────────────────────────────────────────
    metrics  = evaluate(circ_model, test_loader)
    infer_ms = measure_inference_ms(circ_model, DEVICE)
    size_mb  = model_size_mb(circ_model)
    acc_drop = baseline_acc - metrics["accuracy"]

    print("\n  Counting compressed model FLOPs ...", flush=True)
    circ_flops = count_flops(circ_model, DEVICE)
    print(f"  Compressed FLOPs: {circ_flops:,}", flush=True)

    print(f"\n  ✓ Results:")
    print(f"    Accuracy   : {metrics['accuracy']:.4f}  (drop={acc_drop:+.4f})")
    print(f"    Size       : {size_mb:.2f} MB  (original={original_size_mb:.2f} MB)")
    print(f"    Params     : {effective_params:,}  (-{param_reduction*100:.1f}%)")
    print(f"    FLOPs      : {circ_flops:,}")
    print(f"    Inference  : {infer_ms:.1f} ms")

    report = {
        "experiment"        : "weight_sharing_circulant",
        "method"            : "weight_sharing",
        "variant"           : "CirculantMatrices",
        "original_arch"     : MODEL_ARCH,
        "dataset"           : "CIFAR-10",
        "train_device"      : str(DEVICE),
        "block_size"        : BLOCK_SIZE,
        "finetune_epochs"   : FINETUNE_EPOCHS,
        "finetune_lr"       : FINETUNE_LR,
        "method_description": (
            f"Block-circulant weight compression (vectorized): each ({BLOCK_SIZE}×{BLOCK_SIZE}) "
            f"sub-block B of W replaced by circ(c) where c_k = (1/b)Σᵢ B[i,(i+k)%b] "
            f"(diagonal average = LS projection onto circulant subspace). "
            f"Matrix padded to nearest multiple of b; all blocks extracted and projected "
            f"in one batched reshape+gather+mean — no Python loops. "
            f"Conv weights reshaped to 2-D (C_out, C_in·kH·kW) before compression. "
            f"Eval-only weight cache; training rebuilds on every forward pass for "
            f"correct autograd."
        ),
        "compression_time_s"        : round(compress_time, 2),
        "baseline"                  : baseline,
        "accuracy"                  : round(metrics["accuracy"],   6),
        "pre_finetune_accuracy"     : round(pre_ft["accuracy"],    6),
        "accuracy_drop"             : round(acc_drop,              6),
        "finetune_epoch_accs"       : ft_accs,
        "precision"                 : round(metrics["precision"],  6),
        "recall"                    : round(metrics["recall"],     6),
        "f1"                        : round(metrics["f1"],         6),
        "num_parameters"            : effective_params,
        "original_num_parameters"   : original_params,
        "param_reduction_pct"       : round(param_reduction * 100, 2),
        "model_size_mb"             : round(size_mb,               4),
        "original_size_mb"          : round(original_size_mb,      4),
        "compression_ratio"         : round(original_size_mb / size_mb, 4) if size_mb else None,
        "inference_ms"              : round(infer_ms,              4),
        "flops"                     : circ_flops,
        "original_flops"            : original_flops,
        "flops_reduction_pct"       : round((1 - circ_flops / original_flops) * 100, 2)
                                      if original_flops else None,
        "status"                    : "ok",
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved → {OUTPUT_JSON}")


if __name__ == "__main__":
    main()

[init] PyTorch : 2.12.0.dev20260324+cu128
[init] Device  : cuda
[init] BlockSize: 8  FineTuneEpochs: 5

  Weight Sharing — Circulant Structured Matrices
  Model: resnet50  |  BlockSize: 8  |  FT: 5 epochs

  Baseline accuracy: 0.9320

[model] Loading resnet50 from ../__2__baseline_resnet50_cifar10.pth ...
  Original: 90.03 MB  |  Params: 23,520,842

  Counting original FLOPs ...


W0409 12:26:35.914000 38108 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


  Original FLOPs: 2,595,659,776

  Applying Circulant compression ...
    Circ Conv:   (64→64, 1×1)  D=64
    Circ Conv:   (64→64, 3×3)  D=576
    Circ Conv:   (64→256, 1×1)  D=64
    Circ Conv:   (64→256, 1×1)  D=64
    Circ Conv:   (256→64, 1×1)  D=256
    Circ Conv:   (64→64, 3×3)  D=576
    Circ Conv:   (64→256, 1×1)  D=64
    Circ Conv:   (256→64, 1×1)  D=256
    Circ Conv:   (64→64, 3×3)  D=576
    Circ Conv:   (64→256, 1×1)  D=64
    Circ Conv:   (256→128, 1×1)  D=256
    Circ Conv:   (128→128, 3×3)  D=1152
    Circ Conv:   (128→512, 1×1)  D=128
    Circ Conv:   (256→512, 1×1)  D=256
    Circ Conv:   (512→128, 1×1)  D=512
    Circ Conv:   (128→128, 3×3)  D=1152
    Circ Conv:   (128→512, 1×1)  D=128
    Circ Conv:   (512→128, 1×1)  D=512
    Circ Conv:   (128→128, 3×3)  D=1152
    Circ Conv:   (128→512, 1×1)  D=128
    Circ Conv:   (512→128, 1×1)  D=512
    Circ Conv:   (128→128, 3×3)  D=1152
    Circ Conv:   (128→512, 1×1)  D=128
    Circ Conv:   (512→256, 1×1)  D=512
    Circ 